# Clonar TU voz a Piper — 2 celdas
Guion = `corpus/corpus_es_1300.txt`. Grabá un clip por frase: `f00001.wav` = frase 1, etc.
Grabás las que quieras y comprimís la carpeta `wavs/` en **dataset.zip**.

**Antes de empezar:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.
**Celda 1**: subís el zip y prepara. **Celda 2**: entrena.

In [ ]:
#@title 1. SUBIR TUS AUDIOS Y PREPARAR — subís dataset.zip, arma todo — ~15 min
import torch, os, re, wave, glob, zipfile
import numpy as np
assert torch.cuda.is_available(), "Activá T4: Entorno de ejecución -> Cambiar tipo de entorno -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
!wget -q -O /content/corpus.txt "https://raw.githubusercontent.com/Lazy-Money/Loud-Web/claude/readvox-research-slumjx/colab/corpus/corpus_es_1300.txt"
CORPUS = [l.strip() for l in open("/content/corpus.txt", encoding="utf-8") if l.strip()]

from google.colab import files
print("Subí tu dataset.zip (carpeta wavs/ con f00001.wav = frase 1, etc.)")
up = files.upload()
with zipfile.ZipFile(list(up.keys())[0]) as z: z.extractall("/content/unzip")
wavs = glob.glob("/content/unzip/**/*.wav", recursive=True)
print(f"{len(wavs)} audios en el zip. Emparejando con las frases...")

def normw(src, dst):
    with wave.open(src,"rb") as w:
        n,r,ch,sw = w.getnframes(),w.getframerate(),w.getnchannels(),w.getsampwidth()
        raw = w.readframes(n)
    d = np.frombuffer(raw, dtype={1:np.int8,2:np.int16,4:np.int32}[sw]).astype(np.float32)
    if sw!=2: d = d/(2**(8*sw-1))*32767
    if ch>1: d = d.reshape(-1,ch).mean(axis=1)
    if r!=22050:
        idx = np.linspace(0,len(d)-1,int(len(d)*22050/r)); d = np.interp(idx,np.arange(len(d)),d)
    d = np.clip(d,-32768,32767).astype(np.int16)
    with wave.open(dst,"wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050); w.writeframes(d.tobytes())
    return len(d)/22050

os.makedirs("/content/dataset/wavs", exist_ok=True)
rows, total, skip = [], 0.0, 0
for src in sorted(wavs):
    m = re.search(r"f?(\d{3,6})", os.path.basename(src))
    if not m or not (1<=int(m.group(1))<=len(CORPUS)): skip+=1; continue
    idx = int(m.group(1)); n = f"f{idx:05d}"
    seg = normw(src, f"/content/dataset/wavs/{n}.wav")
    if not 1.0<=seg<=20.0: skip+=1; continue
    rows.append(f"{n}|{CORPUS[idx-1]}"); total+=seg
open("/content/dataset/metadata.csv","w",encoding="utf-8").write("\n".join(rows)+"\n")
assert rows, "Ningún audio se pudo emparejar. Deben llamarse f00001.wav, f00002.wav..."
print(f"Dataset: {len(rows)} clips, {total/60:.1f} min de tu voz. Instalando Piper...")

%cd /content
!git clone -q https://github.com/rhasspy/piper.git
%cd /content/piper/src/python
!pip install -q -e . "pytorch-lightning~=1.9" espeak-phonemizer librosa "numpy<2"
!apt-get install -yq espeak-ng > /dev/null
!bash build_monotonic_align.sh > /dev/null 2>&1
!python -m piper_train.preprocess --language es --input-dir /content/dataset --output-dir /content/train_out --dataset-format ljspeech --single-speaker --sample-rate 22050
!wget -q -nc -O /content/base.ckpt "https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_ES/davefx/medium/epoch%3D2218-step%3D562840.ckpt"
print("\n=== LISTO. Ahora corré la celda 2 para entrenar. ===")

In [ ]:
#@title 2. ENTRENAR Y DESCARGAR — 2-4 h (podés cortarla y reejecutarla para exportar lo entrenado)
import glob, shutil, os
%cd /content/piper/src/python
ck = sorted(glob.glob("/content/train_out/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)
if not ck:
    !python -m piper_train --dataset-dir /content/train_out --accelerator gpu --devices 1 --batch-size 16 --validation-split 0.0 --num-test-examples 0 --max_epochs 3219 --resume_from_checkpoint /content/base.ckpt --checkpoint-epochs 5 --precision 32 --quality medium
    ck = sorted(glob.glob("/content/train_out/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)
assert ck, "No hay checkpoint todavía: dejá entrenar unos minutos y reejecutá esta celda."
!python -m piper_train.export_onnx "{ck[-1]}" /content/mi_voz.onnx
shutil.copy("/content/train_out/config.json", "/content/mi_voz.onnx.json")
from google.colab import files
files.download("/content/mi_voz.onnx")
files.download("/content/mi_voz.onnx.json")
print("Copiá ambos a tu carpeta de voces de LoudVox y elegí 'mi_voz' en Configuración")